In [1]:
import sys
import os
import matplotlib.pyplot as plt
import cv2
import numpy as np

# Add the src directory to the path. TEMPORARY FIX
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))

from src.data_processing.dataset_loader import CoastData

In [2]:
data_path = os.path.abspath(os.path.join(os.getcwd(), "../../data/SCLabels_v1.0.0/"))

# Load the data, all the different stations
data = CoastData(data_path)
station_names = data.get_station_names()

CLASS_MAP = {
    0: "NoData",
    25: "NotClassified",
    75: "Landwards",
    150: "Seawards",
    255: "Shoreline"
}

total_pixels_station_and_class = {station: {cls: 0 for cls in CLASS_MAP.keys()} for station in station_names} 
total_pixels_station = {station: 0 for station in station_names}
total_pixels_global = {cls: 0 for cls in CLASS_MAP.keys()}
print(total_pixels_station_and_class)

station_shapes = {station: {'heights': [], 'widths': []} for station in station_names}

CoastData: global - 1717 images
{'agrelo': {0: 0, 25: 0, 75: 0, 150: 0, 255: 0}, 'arenaldentem': {0: 0, 25: 0, 75: 0, 150: 0, 255: 0}, 'cadiz': {0: 0, 25: 0, 75: 0, 150: 0, 255: 0}, 'cies': {0: 0, 25: 0, 75: 0, 150: 0, 255: 0}, 'samarador': {0: 0, 25: 0, 75: 0, 150: 0, 255: 0}}


In [3]:
for station in station_names:
    # print(station)
    station_data = data.get_images(station)
    print(f"\nStation: {station}")
    print(f"Number of images in station {station}: {len(station_data)}")

    for image_data in station_data:

        mask = cv2.imread(image_data['mask'], cv2.IMREAD_GRAYSCALE)
        total_pixels = mask.shape[0] * mask.shape[1]
        total_pixels_station[station] += total_pixels
        
        classes, count = np.unique(mask, return_counts=True)
        for cls, cnt in zip(classes, count):
            percentage = (cnt / total_pixels) * 100
            total_pixels_station_and_class[station][cls] += cnt
            total_pixels_global[cls] += cnt
            
        
        height, width = mask.shape
        station_shapes[station]['heights'].append(height)
        station_shapes[station]['widths'].append(width)

    for cls, cnt in total_pixels_station_and_class[station].items():
        percentage = (cnt / total_pixels_station[station]) * 100 if total_pixels_station[station] > 0 else 0
        print(f"Class {CLASS_MAP[cls]}: {cnt} pixels ({percentage:.2f}%)")

    heights = station_shapes[station]['heights']
    widths = station_shapes[station]['widths']
    print(f"Image heights in station {station}: min={min(heights)}, max={max(heights)}, mean={np.mean(heights):.2f}, std={np.std(heights):.2f}")
    print(f"Image widths in station {station}: min={min(widths)}, max={max(widths)}, mean={np.mean(widths):.2f}, std={np.std(widths):.2f}")

print("\nGlobal statistics:")
for cls, cnt in total_pixels_global.items():
    total_pixels = sum(total_pixels_global.values())
    percentage = (cnt / total_pixels) * 100 if total_pixels > 0 else 0
    print(f"Class {CLASS_MAP[cls]}: {cnt} pixels ({percentage:.2f}%)")



Station: agrelo
Number of images in station agrelo: 244
Class NoData: 19411234 pixels (22.55%)
Class NotClassified: 3008998 pixels (3.50%)
Class Landwards: 44377212 pixels (51.56%)
Class Seawards: 19067447 pixels (22.15%)
Class Shoreline: 206564 pixels (0.24%)
Image heights in station agrelo: min=320, max=473, mean=440.39, std=17.12
Image widths in station agrelo: min=801, max=801, mean=801.00, std=0.00

Station: arenaldentem
Number of images in station arenaldentem: 40
Class NoData: 1982626 pixels (17.75%)
Class NotClassified: 276469 pixels (2.47%)
Class Landwards: 6559062 pixels (58.71%)
Class Seawards: 2325954 pixels (20.82%)
Class Shoreline: 28237 pixels (0.25%)
Image heights in station arenaldentem: min=333, max=371, mean=348.70, std=8.60
Image widths in station arenaldentem: min=801, max=801, mean=801.00, std=0.00

Station: cadiz
Number of images in station cadiz: 946
Class NoData: 153406676 pixels (29.37%)
Class NotClassified: 9731730 pixels (1.86%)
Class Landwards: 169430075 p